In [1]:
# Define column names
column_names = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted',
    'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'class', 'difficulty'
]

In [2]:
import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# Load the training data
train_df = pd.read_csv(
    r"C:\Users\yasha\Downloads\nsl-kdd\KDDTrain+.txt",
    header=None,
    names=column_names
)

# Drop the 'difficulty' column
train_df.drop(['difficulty'], axis=1, inplace=True)

# Load the testing data
test_df = pd.read_csv(
    r"C:\Users\yasha\Downloads\nsl-kdd\KDDTest+.txt",
    header=None,
    names=column_names
)

# Drop the 'difficulty' column
test_df.drop(['difficulty'], axis=1, inplace=True)

# Print basic info
print("Training Data")
print(f"Shape: {train_df.shape}")
print(train_df.head())

print("\n--- Testing Data ---")
print(f"Shape: {test_df.shape}")
print(test_df.head())


Training Data
Shape: (125973, 42)
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2         0           tcp   private   S0          0          0     0   
3         0           tcp      http   SF        232       8153     0   
4         0           tcp      http   SF        199        420     0   

   wrong_fragment  urgent  hot  ...  dst_host_srv_count  \
0               0       0    0  ...                  25   
1               0       0    0  ...                   1   
2               0       0    0  ...                  26   
3               0       0    0  ...                 255   
4               0       0    0  ...                 255   

   dst_host_same_srv_rate  dst_host_diff_srv_rate  \
0                    0.17                    0.03   
1                    0.00                    0.60   
2                  

In [3]:
# Create a binary classification target
train_df['attack_binary'] = train_df['class'].apply(lambda x: 0 if x == 'normal' else 1)
test_df['attack_binary']  = test_df['class'].apply(lambda x: 0 if x == 'normal' else 1)


In [4]:
# Separate features (X) and target (y)
X_train_raw = train_df.drop(['class', 'attack_binary'], axis=1)
y_train = train_df['attack_binary']

X_test_raw = test_df.drop(['class', 'attack_binary'], axis=1)
y_test = test_df['attack_binary']

# Identify categorical and numerical columns
categorical_cols = ['protocol_type', 'service', 'flag']
numerical_cols = X_train_raw.columns.drop(categorical_cols)

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols.tolist())


Categorical Columns: ['protocol_type', 'service', 'flag']
Numerical Columns: ['duration', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']


In [5]:
# One-Hot Encode categorical features
X_train_encoded = pd.get_dummies(X_train_raw, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, columns=categorical_cols, drop_first=True)

# Align columns to ensure test set has same features as train set
train_cols = X_train_encoded.columns
test_cols = X_test_encoded.columns

# Add missing columns in test set
missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    X_test_encoded[c] = 0

# Add missing columns in train set
missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    X_train_encoded[c] = 0

# To ensure same order in both train and test sets
X_test_encoded = X_test_encoded[train_cols]

print(f"Shape of training data after encoding: {X_train_encoded.shape}")
print(f"Shape of testing data after encoding: {X_test_encoded.shape}")


Shape of training data after encoding: (125973, 119)
Shape of testing data after encoding: (22544, 119)


In [6]:
from sklearn.preprocessing import StandardScaler

# Identify the new numerical columns (original ones)
numerical_cols_to_scale = numerical_cols  

# Fit on training data and transform both train and test dataw
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

X_train_scaled[numerical_cols_to_scale] = StandardScaler().fit_transform(X_train_encoded[numerical_cols_to_scale])
X_test_scaled[numerical_cols_to_scale] = StandardScaler().fit(X_train_encoded[numerical_cols_to_scale]).transform(X_test_encoded[numerical_cols_to_scale])

print("\n--- Scaled Training Data Head ---")
print(X_train_scaled.head())




--- Scaled Training Data Head ---
   duration  src_bytes  dst_bytes      land  wrong_fragment    urgent  \
0 -0.110249  -0.007679  -0.004919 -0.014089       -0.089486 -0.007736   
1 -0.110249  -0.007737  -0.004919 -0.014089       -0.089486 -0.007736   
2 -0.110249  -0.007762  -0.004919 -0.014089       -0.089486 -0.007736   
3 -0.110249  -0.007723  -0.002891 -0.014089       -0.089486 -0.007736   
4 -0.110249  -0.007728  -0.004814 -0.014089       -0.089486 -0.007736   

        hot  num_failed_logins  logged_in  num_compromised  ...  flag_REJ  \
0 -0.095076          -0.027023  -0.809262        -0.011664  ...     False   
1 -0.095076          -0.027023  -0.809262        -0.011664  ...     False   
2 -0.095076          -0.027023  -0.809262        -0.011664  ...     False   
3 -0.095076          -0.027023   1.235694        -0.011664  ...     False   
4 -0.095076          -0.027023   1.235694        -0.011664  ...     False   

   flag_RSTO  flag_RSTOS0  flag_RSTR  flag_S0  flag_S1  flag_S2

In [7]:
#Feature Scaling
X_train_final = X_train_scaled.copy()
X_test_final = X_test_scaled.copy()

print("Final Train Features:", X_train_final.shape)
print("Final Train Labels:", y_train.shape)
print("Final Test Features:", X_test_final.shape)
print("Final Test Labels:", y_test.shape)

Final Train Features: (125973, 119)
Final Train Labels: (125973,)
Final Test Features: (22544, 119)
Final Test Labels: (22544,)


In [8]:
%pip install imblearn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.impute import SimpleImputer

print("Before SMOTE:", Counter(y_train))

# Handle missing values before applying SMOTE
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train_scaled)

# Apply SMOTE only on training data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_imputed, y_train)

#class distribution after SMOTE
print("After SMOTE:", Counter(y_train_res))

print("Original training shape:", X_train_final.shape, y_train.shape)
print("Resampled training shape:", X_train_res.shape, y_train_res.shape)

Before SMOTE: Counter({0: 67343, 1: 58630})
After SMOTE: Counter({0: 67343, 1: 67343})
Original training shape: (125973, 119) (125973,)
Resampled training shape: (134686, 119) (134686,)


In [ ]:
# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LogisticRegression
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# from sklearn.impute import SimpleImputer
# import time # Added to show training time, which is good practice

# # --- 1. Define Models for Comparison ---
# models = {
#     "Logistic Regression": LogisticRegression(max_iter=2000, solver='lbfgs', n_jobs=1, random_state=42),
#     "Decision Tree": DecisionTreeClassifier(max_depth=None, random_state=42),
#     "Random Forest (300 est.)": RandomForestClassifier(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1),
#     "Gradient Boosting": GradientBoostingClassifier(random_state=42)
# }

# # --- 2. Train and Evaluate Models for Comparison ---
# print("--- Model Comparison Phase ---")
# for name, model in models.items():
#     print(f"Training {name}...")
#     start_time = time.time()
#     model.fit(X_train_scaled, y_train)
#     end_time = time.time()
#     print(f"{name} trained in: {(end_time - start_time):.2f} seconds")

# # Evaluate the models
# for name, model in models.items():
#     print(f"\nEvaluating {name}...")
#     # Use y_pred for the results of the model comparison loop
#     y_pred = model.predict(X_test_scaled) 
    
#     accuracy = accuracy_score(y_test, y_pred)
#     print(f"{name} Accuracy: {accuracy * 100:.2f}%")
#     print(f"\n{name} Classification Report:")
#     print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

# # --- 3. Individual Random Forest for Feature Selection (Corrected Evaluation) ---
# rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)

# print("\n--- Individual RF Model (n_estimators=100) for Feature Analysis ---")
# print("Training Random Forest Model")
# start_time = time.time()
# rf_model.fit(X_train_scaled, y_train)
# end_time = time.time()
# print(f"Training completed in: {(end_time - start_time):.2f} seconds")

# # Make predictions using this specific rf_model
# pred_fs = rf_model.predict(X_test_scaled)

# # The evaluation metrics now correctly use the predictions from this model (pred_fs)
# accuracy_fs = accuracy_score(y_test, pred_fs)
# print(f"\nModel Accuracy (RF n=100): {accuracy_fs * 100:.2f}%")

# print("\nClassification Report (RF n=100):")
# print(classification_report(y_test, pred_fs, target_names=['Normal', 'Attack']))

# # --- Example of Feature Selection Use ---
# # You can now access the feature importances for this specific model:
# # feature_importances = rf_model.feature_importances_
# # print("\nFeature Importances:", feature_importances)

In [11]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.1.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB 
from sklearn.svm import LinearSVC

models = {
    # Linear Models
    "Logistic Regression": LogisticRegression(max_iter=2000, solver='lbfgs', n_jobs=1, random_state=42),
    "Linear SVM": LinearSVC(max_iter=500, random_state=42, dual=False), # Added Linear SVM, max_iter low for speed/stability
    
    # Tree-Based Models
    "Decision Tree": DecisionTreeClassifier(max_depth=None, random_state=42),
    "Random Forest (150 est.)": RandomForestClassifier(n_estimators=150, max_depth=None, random_state=42, n_jobs=1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost Classifier": XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', n_jobs=1, random_state=42),
    
    # Other Classifiers
    "K-Nearest Neighbors (k=5)": KNeighborsClassifier(n_neighbors=5, n_jobs=1),
    "Gaussian Naive Bayes": GaussianNB() # Added Gaussian Naive Bayes
}

# --- 2. Train and Evaluate ALL Models ---
print("\n--- Model Comparison Phase (8 Models) ---")
for name, model in models.items():
    print(f"Training {name}...")
    start_time = time.time()
    model.fit(X_train_scaled, y_train) 
    end_time = time.time()
    print(f"{name} trained in: {(end_time - start_time):.2f} seconds")

# Evaluate the models
target_names = ['Normal', 'Attack'] 
for name, model in models.items():
    print(f"\nEvaluating {name}...")
    # Gaussian Naive Bayes expects non-negative input, but it often works okay with scaled data.
    y_pred = model.predict(X_test_scaled)
    
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name} Accuracy: {accuracy * 100:.2f}%")
    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, y_pred, target_names=target_names))

# --- 3. Feature Importance Analysis (Using Random Forest as the baseline) ---
# Kept n_jobs=1 for stability
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)

print("\n--- Feature Importance Analysis (RF n=100) ---")
print("Training Random Forest Model")
start_time = time.time()
rf_model.fit(X_train_scaled, y_train)
end_time = time.time()
print(f"Training completed in: {(end_time - start_time):.2f} seconds")

# Make predictions using this specific rf_model
pred_fs = rf_model.predict(X_test_scaled)

accuracy_fs = accuracy_score(y_test, pred_fs)
print(f"\nModel Accuracy (RF n=100): {accuracy_fs * 100:.2f}%")

print("\nClassification Report (RF n=100):")
print(classification_report(y_test, pred_fs, target_names=target_names))

# Feature Importances
feature_importances = rf_model.feature_importances_
print("\nFeature Importances (Top 5):")
feature_names = X_train_scaled.columns
feature_indices = np.argsort(feature_importances)[::-1][:5]
for i in feature_indices:
    print(f"{feature_names[i]}: {feature_importances[i]:.4f}")


--- Model Comparison Phase (8 Models) ---
Training Logistic Regression...
Logistic Regression trained in: 4.97 seconds
Training Linear SVM...
Linear SVM trained in: 4.77 seconds
Training Decision Tree...
Decision Tree trained in: 1.05 seconds
Training Random Forest (150 est.)...
Random Forest (150 est.) trained in: 11.50 seconds
Training Gradient Boosting...
Gradient Boosting trained in: 25.88 seconds
Training XGBoost Classifier...


c:\Users\yasha\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:53:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Classifier trained in: 2.74 seconds
Training K-Nearest Neighbors (k=5)...
K-Nearest Neighbors (k=5) trained in: 0.17 seconds
Training Gaussian Naive Bayes...
Gaussian Naive Bayes trained in: 0.48 seconds

Evaluating Logistic Regression...
Logistic Regression Accuracy: 74.40%

Logistic Regression Classification Report:
              precision    recall  f1-score   support

      Normal       0.64      0.93      0.76      9711
      Attack       0.91      0.61      0.73     12833

    accuracy                           0.74     22544
   macro avg       0.78      0.77      0.74     22544
weighted avg       0.80      0.74      0.74     22544


Evaluating Linear SVM...
Linear SVM Accuracy: 74.43%

Linear SVM Classification Report:
              precision    recall  f1-score   support

      Normal       0.64      0.93      0.76      9711
      Attack       0.91      0.61      0.73     12833

    accuracy                           0.74     22544
   macro avg       0.78      0.77     